# Limpieza y preprocesamiento — NYC Yellow Taxi 2022 + 2023

Este notebook hace todo solo:

1. **Descarga** los 24 archivos mensuales de la TLC (enero 2022 a diciembre 2023) en `data/raw/`. Si un mes ya está descargado, lo salta.
2. **Limpia cada mes** con las mismas reglas acordadas por el equipo (versión 2).
3. **Junta todo en un solo archivo:** `data/processed/yellow_2022_2023_clean.parquet`.

Procesa mes por mes, así que no necesita cargar los dos años completos en memoria.

## Qué tiene el archivo final

| Columna | Qué es |
|---|---|
| `tpep_pickup_datetime` | Fecha y hora de recojo (para dividir train / val / test por fechas) |
| `PULocationID`, `DOLocationID`, `pickup_hour`, `pickup_dow`, `is_airport_trip`, `morning_rush`, `night_rush`, `weekend_rush` | **Los 8 inputs del modelo** |
| `fare_amount` | Target de tarifa (USD) |
| `trip_duration_min` | Target de duración (minutos) |
| `valid_for_fare` | `True` si la fila también pasa las reglas de tarifa. **Modelo de tarifa: usar solo estas filas** |
| `new_tariff` | 1 si el viaje es desde el 19-dic-2022 (tarifa nueva de la TLC) |
| `trip_distance` | Distancia real. Se conserva, pero **no es input del modelo** (data leakage) |

Un solo archivo sirve para los dos modelos:
- **Duración:** todas las filas.
- **Tarifa:** solo las filas con `valid_for_fare == True`.

## Reglas

| Regla | Duración | Tarifa |
|---|:---:|:---:|
| Fecha de recojo fuera del mes del archivo | ✅ | ✅ |
| Filas completamente duplicadas | ✅ | ✅ |
| `duration_min` ≤ 1 o > 180 | ✅ | ✅ |
| `trip_distance` < 0.1 o > 100 millas | ✅ | ✅ |
| `avg_speed_mph` < 1 o > 80 | ✅ | ✅ |
| `RatecodeID` = 5 (tarifa negociada) | — | ✅ |
| `fare_amount` menor que la bajada de bandera | — | ✅ |
| `fare_amount` > 500 | — | ✅ |

No se eliminan las filas sin metadatos ni las zonas 264/265, y no hay topes por percentil.

## Cambio de tarifa de la TLC (importante para el modelo de tarifa)

Desde el **19 de diciembre de 2022** la bajada de bandera pasó de **$2.50 a $3.00**, cada unidad de $0.50 a $0.70 y la tarifa plana a JFK de $52 a $70.

- La regla del piso usa **$2.50 antes** y **$3.00 desde** esa fecha.
- La columna `new_tariff` marca el periodo. Sin ella, el modelo de tarifa mezclaría precios de dos tarifas distintas. **El equipo decide si la usa como feature.**

## No se sube a GitHub

`data/raw/` (unos 24 archivos) y el parquet final pesan cientos de MB, más que el límite de 100 MB de GitHub. Se suben **solo este notebook** y `limpieza_log_2022_2023.csv`. El parquet se comparte por Drive, o cada uno lo genera ejecutando este notebook.

## 1. Configuración

In [1]:
import shutil
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Raíz del proyecto = carpeta que contiene /data
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_PATH = OUT_DIR / "yellow_2022_2023_clean.parquet"
LOG_PATH = OUT_DIR / "limpieza_log_2022_2023.csv"

PERIODOS = [(anio, mes) for anio in (2022, 2023) for mes in range(1, 13)]
URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{anio}-{mes:02d}.parquet"
BORRAR_CRUDOS_AL_FINAL = False   # True: borra data/raw/ al terminar para liberar espacio

# Reglas acordadas (v2)
DURACION_MIN, DURACION_MAX = 1, 180        # se elimina si <= 1 o > 180 min
DISTANCIA_MIN, DISTANCIA_MAX = 0.1, 100    # se elimina si < 0.1 o > 100 millas
VELOCIDAD_MIN, VELOCIDAD_MAX = 1, 80       # se elimina si < 1 o > 80 mph
RATECODE_EXCLUIDO = 5                      # solo tarifa
TARIFA_MAX = 500                           # solo tarifa

# Cambio de tarifa TLC: piso de la bajada de bandera según la fecha
INICIO_TARIFA_NUEVA = pd.Timestamp("2022-12-19")
TARIFA_MIN_ANTES, TARIFA_MIN_DESDE = 2.50, 3.00

# Features (iguales a build_features() de la API)
ZONAS_AEROPUERTO = [1, 132, 138]
HORAS_MORNING_RUSH = [7, 8, 9]            # lun–vie 07:00–09:59
HORAS_NIGHT_RUSH = [17, 18, 19]           # lun–vie 17:00–19:59
HORAS_WEEKEND_RUSH = list(range(10, 20))  # sáb–dom 10:00–19:59

SCHEMA = pa.schema([
    ("tpep_pickup_datetime", pa.timestamp("us")),
    ("trip_distance", pa.float32()),
    ("PULocationID", pa.int16()),
    ("DOLocationID", pa.int16()),
    ("pickup_hour", pa.int8()),
    ("pickup_dow", pa.int8()),
    ("is_airport_trip", pa.int8()),
    ("morning_rush", pa.int8()),
    ("night_rush", pa.int8()),
    ("weekend_rush", pa.int8()),
    ("fare_amount", pa.float32()),
    ("trip_duration_min", pa.float32()),
    ("valid_for_fare", pa.bool_()),
    ("new_tariff", pa.int8()),
])

print("ROOT:", ROOT)
print("Meses a procesar:", len(PERIODOS), f"({PERIODOS[0][0]}-{PERIODOS[0][1]:02d} a {PERIODOS[-1][0]}-{PERIODOS[-1][1]:02d})")

ROOT: c:\Users\FRANK FLORES\AnyoneAI_Proyect
Meses a procesar: 24 (2022-01 a 2023-12)


## 2. Descarga automática

Descarga cada mes a `data/raw/`. Si se corta el internet, vuelve a ejecutar la celda: los meses que ya están completos se saltan.

In [2]:
def ruta_cruda(anio: int, mes: int) -> Path:
    return RAW_DIR / f"yellow_tripdata_{anio}-{mes:02d}.parquet"


def es_parquet_valido(ruta: Path) -> bool:
    try:
        pq.ParquetFile(ruta).metadata
        return True
    except Exception:
        return False


def descargar(anio: int, mes: int, intentos: int = 3) -> str:
    destino = ruta_cruda(anio, mes)
    if destino.exists() and es_parquet_valido(destino):
        return "ya estaba"
    url = URL.format(anio=anio, mes=mes)
    parcial = destino.with_name(destino.name + ".part")
    for intento in range(1, intentos + 1):
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(pedido, timeout=120) as resp, open(parcial, "wb") as f:
                shutil.copyfileobj(resp, f, length=1024 * 1024)
            if not es_parquet_valido(parcial):
                raise ValueError("el archivo descargado está incompleto")
            parcial.replace(destino)
            return "descargado"
        except Exception as error:
            print(f"   intento {intento}/{intentos} falló: {error}")
            parcial.unlink(missing_ok=True)
            time.sleep(5 * intento)
    raise RuntimeError(f"No se pudo descargar {url}. Revisa tu conexión y vuelve a ejecutar la celda.")


RAW_DIR.mkdir(parents=True, exist_ok=True)
inicio = time.time()
for anio, mes in PERIODOS:
    estado = descargar(anio, mes)
    print(f"{anio}-{mes:02d}: {estado:<10} {ruta_cruda(anio, mes).stat().st_size / 1e6:6.1f} MB")
total_mb = sum(ruta_cruda(a, m).stat().st_size for a, m in PERIODOS) / 1e6
print(f"\nListo: {len(PERIODOS)} archivos, {total_mb:,.0f} MB en total ({time.time() - inicio:.0f} s)")

   intento 1/3 falló: <urlopen error [WinError 10054] Se ha forzado la interrupción de una conexión existente por el host remoto>
2022-01: descargado   38.1 MB
2022-02: descargado   45.6 MB
2022-03: descargado   55.7 MB
2022-04: descargado   55.2 MB
2022-05: descargado   55.6 MB
2022-06: descargado   55.4 MB
   intento 1/3 falló: <urlopen error [WinError 10054] Se ha forzado la interrupción de una conexión existente por el host remoto>
2022-07: descargado   49.4 MB
2022-08: descargado   49.7 MB
2022-09: descargado   49.6 MB
2022-10: descargado   57.1 MB
2022-11: descargado   50.1 MB
2022-12: descargado   53.6 MB
2023-01: descargado   47.7 MB
2023-02: descargado   47.7 MB
2023-03: descargado   56.1 MB
2023-04: descargado   54.2 MB
2023-05: descargado   58.7 MB
2023-06: descargado   55.0 MB
2023-07: descargado   48.4 MB
2023-08: descargado   48.2 MB
2023-09: descargado   47.9 MB
2023-10: descargado   59.0 MB
2023-11: descargado   56.1 MB
2023-12: descargado   56.8 MB

Listo: 24 archivos,

## 3. Limpieza de un mes

Cada regla cuenta solo las filas que las anteriores no quitaron. Las reglas de tarifa **no borran filas**: solo marcan `valid_for_fare`.

In [3]:
def limpiar_mes(anio: int, mes: int) -> tuple[pd.DataFrame, list[dict], int]:
    df = pd.read_parquet(ruta_cruda(anio, mes))
    n_crudo = len(df)
    periodo = f"{anio}-{mes:02d}"
    inicio_mes = pd.Timestamp(anio, mes, 1)
    fin_mes = inicio_mes + pd.offsets.MonthBegin(1)

    pickup = df["tpep_pickup_datetime"]
    t = (df["tpep_dropoff_datetime"] - pickup).dt.total_seconds() / 60
    d = df["trip_distance"]
    v = d / (t / 60)

    reglas = {
        "Fecha fuera del mes": ~pickup.between(inicio_mes, fin_mes, inclusive="left"),
        "Filas completamente duplicadas": df.duplicated(),
        f"duration_min <= {DURACION_MIN} o > {DURACION_MAX}": ~((t > DURACION_MIN) & (t <= DURACION_MAX)),
        f"trip_distance < {DISTANCIA_MIN} o > {DISTANCIA_MAX}": ~((d >= DISTANCIA_MIN) & (d <= DISTANCIA_MAX)),
        f"avg_speed_mph < {VELOCIDAD_MIN} o > {VELOCIDAD_MAX}": ~((v >= VELOCIDAD_MIN) & (v <= VELOCIDAD_MAX)),
    }
    conservar = pd.Series(True, index=df.index)
    log = []
    for nombre, eliminar in reglas.items():
        nuevos = eliminar & conservar
        log.append({"periodo": periodo, "tipo": "elimina fila", "regla": nombre, "filas": int(nuevos.sum())})
        conservar &= ~eliminar

    out = pd.DataFrame({
        "tpep_pickup_datetime": pickup[conservar].astype("datetime64[us]"),
        "trip_distance": d[conservar].astype("float32"),
        "PULocationID": df.loc[conservar, "PULocationID"].astype("int16"),
        "DOLocationID": df.loc[conservar, "DOLocationID"].astype("int16"),
    })
    hora = out["tpep_pickup_datetime"].dt.hour
    dia = out["tpep_pickup_datetime"].dt.dayofweek
    laborable = dia < 5
    out["pickup_hour"] = hora.astype("int8")
    out["pickup_dow"] = dia.astype("int8")
    out["is_airport_trip"] = (out["PULocationID"].isin(ZONAS_AEROPUERTO) | out["DOLocationID"].isin(ZONAS_AEROPUERTO)).astype("int8")
    out["morning_rush"] = (laborable & hora.isin(HORAS_MORNING_RUSH)).astype("int8")
    out["night_rush"] = (laborable & hora.isin(HORAS_NIGHT_RUSH)).astype("int8")
    out["weekend_rush"] = (~laborable & hora.isin(HORAS_WEEKEND_RUSH)).astype("int8")
    out["fare_amount"] = df.loc[conservar, "fare_amount"].astype("float32")
    out["trip_duration_min"] = t[conservar].astype("float32")

    # Reglas de tarifa: marcan, no borran
    nueva = out["tpep_pickup_datetime"] >= INICIO_TARIFA_NUEVA
    piso = np.where(nueva, TARIFA_MIN_DESDE, TARIFA_MIN_ANTES)
    fare = out["fare_amount"]
    reglas_tarifa = {
        f"RatecodeID = {RATECODE_EXCLUIDO}": (df.loc[conservar, "RatecodeID"] == RATECODE_EXCLUIDO).to_numpy(),
        "fare_amount < bajada de bandera ($2.50 / $3.00)": ~(fare.to_numpy() >= piso),
        f"fare_amount > {TARIFA_MAX}": ~(fare.to_numpy() <= TARIFA_MAX),
    }
    valida = np.ones(len(out), dtype=bool)
    for nombre, invalida in reglas_tarifa.items():
        log.append({"periodo": periodo, "tipo": "solo tarifa (marca)", "regla": nombre, "filas": int((invalida & valida).sum())})
        valida &= ~invalida
    out["valid_for_fare"] = valida
    out["new_tariff"] = nueva.astype("int8")
    out = out.sort_values("tpep_pickup_datetime", kind="stable").reset_index(drop=True)

    # Verificaciones del mes
    assert out.notna().all().all(), f"{periodo}: hay nulos"
    assert out["tpep_pickup_datetime"].between(inicio_mes, fin_mes, inclusive="left").all()
    assert (out["trip_duration_min"] > DURACION_MIN).all() and (out["trip_duration_min"] <= DURACION_MAX).all()
    assert out["trip_distance"].between(DISTANCIA_MIN - 1e-6, DISTANCIA_MAX).all()
    assert list(out.columns) == SCHEMA.names
    return out, log, n_crudo

## 4. Procesar los 24 meses y escribir un solo parquet

Se escribe mes por mes en el mismo archivo. Primero va a un `.part` y al final se renombra, así nunca queda un archivo a medias con el nombre final.

In [4]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
parcial = OUT_PATH.with_name(OUT_PATH.name + ".part")
logs, resumen = [], []
inicio = time.time()

with pq.ParquetWriter(parcial, SCHEMA, compression="zstd") as writer:
    for anio, mes in PERIODOS:
        t0 = time.time()
        out, log, n_crudo = limpiar_mes(anio, mes)
        writer.write_table(pa.Table.from_pandas(out, schema=SCHEMA, preserve_index=False), row_group_size=1_000_000)
        logs.extend(log)
        validas = out["valid_for_fare"]
        resumen.append({
            "periodo": f"{anio}-{mes:02d}",
            "anio": anio,
            "crudo": n_crudo,
            "limpio_duracion": len(out),
            "limpio_tarifa": int(validas.sum()),
            "tarifa_media": float(out.loc[validas, "fare_amount"].mean()),
            "duracion_media": float(out["trip_duration_min"].mean()),
            "pct_tarifa_nueva": float(out["new_tariff"].mean() * 100),
        })
        print(f"{anio}-{mes:02d}: {n_crudo:>10,} crudos → {len(out):>10,} limpios ({len(out) / n_crudo:6.1%}) "
              f"| válidos tarifa {validas.mean():6.1%} | {time.time() - t0:4.0f} s")
        del out

parcial.replace(OUT_PATH)
pd.DataFrame(logs).to_csv(LOG_PATH, index=False)
print(f"\nArchivo: {OUT_PATH.relative_to(ROOT)} ({OUT_PATH.stat().st_size / 1e6:,.0f} MB) en {(time.time() - inicio) / 60:.1f} min")
print(f"Log:     {LOG_PATH.relative_to(ROOT)}")

2022-01:  2,463,931 crudos →  2,417,011 limpios ( 98.1%) | válidos tarifa  99.4% |   11 s
2022-02:  2,979,431 crudos →  2,926,470 limpios ( 98.2%) | válidos tarifa  99.5% |   12 s
2022-03:  3,627,882 crudos →  3,560,617 limpios ( 98.1%) | válidos tarifa  99.4% |   18 s
2022-04:  3,599,920 crudos →  3,531,251 limpios ( 98.1%) | válidos tarifa  99.4% |   17 s
2022-05:  3,588,295 crudos →  3,513,268 limpios ( 97.9%) | válidos tarifa  99.3% |   16 s
2022-06:  3,558,124 crudos →  3,474,556 limpios ( 97.7%) | válidos tarifa  99.2% |   16 s
2022-07:  3,174,394 crudos →  3,104,150 limpios ( 97.8%) | válidos tarifa  99.2% |   13 s
2022-08:  3,152,677 crudos →  3,080,628 limpios ( 97.7%) | válidos tarifa  99.1% |   12 s
2022-09:  3,183,767 crudos →  3,108,856 limpios ( 97.6%) | válidos tarifa  99.1% |   13 s
2022-10:  3,675,411 crudos →  3,585,624 limpios ( 97.6%) | válidos tarifa  99.1% |   17 s
2022-11:  3,252,717 crudos →  3,166,989 limpios ( 97.4%) | válidos tarifa  99.1% |   14 s
2022-12:  

## 5. Verificación y resumen

In [5]:
pf = pq.ParquetFile(OUT_PATH)
assert pf.schema_arrow.equals(SCHEMA), "El esquema no coincide"
res = pd.DataFrame(resumen)
assert pf.metadata.num_rows == res["limpio_duracion"].sum(), "No coincide el número de filas"

total = res[["crudo", "limpio_duracion", "limpio_tarifa"]].sum()
print(f"Filas en el archivo: {pf.metadata.num_rows:,} | grupos de filas: {pf.metadata.num_row_groups}")
print(f"Crudo {total['crudo']:,} → duración {total['limpio_duracion']:,} ({total['limpio_duracion'] / total['crudo']:.1%})"
      f" | tarifa {total['limpio_tarifa']:,} ({total['limpio_tarifa'] / total['crudo']:.1%})")

por_anio = res.groupby("anio").agg(
    crudo=("crudo", "sum"),
    limpio_duracion=("limpio_duracion", "sum"),
    limpio_tarifa=("limpio_tarifa", "sum"),
)
por_anio["%_conservado"] = (por_anio["limpio_duracion"] / por_anio["crudo"] * 100).round(2)
por_anio

Filas en el archivo: 76,079,181 | grupos de filas: 90
Crudo 77,966,324 → duración 76,079,181 (97.6%) | tarifa 75,388,386 (96.7%)


,crudo,limpio_duracion,limpio_tarifa,%_conservado
anio,,,,
2022,39656098,38781001,38481896,97.79
2023,38310226,37298180,36906490,97.36


In [6]:
# Tarifa y duración promedio por mes: debería verse el salto de tarifa desde diciembre 2022
res[["periodo", "limpio_duracion", "limpio_tarifa", "tarifa_media", "duracion_media", "pct_tarifa_nueva"]].round(2)

,periodo,limpio_duracion,limpio_tarifa,tarifa_media,duracion_media,pct_tarifa_nueva
0,2022-01,2417011,2402841,12.61,12.87,0.00
1,2022-02,2926470,2910448,13.00,14.24,0.00
2,2022-03,3560617,3538969,13.75,15.00,0.00
3,2022-04,3531251,3509228,14.23,15.61,0.00
4,2022-05,3513268,3488269,14.91,16.61,0.00
5,2022-06,3474556,3448286,14.74,16.57,0.00
6,2022-07,3104150,3078535,14.45,15.35,0.00
7,2022-08,3080628,3053515,14.53,15.57,0.00
8,2022-09,3108856,3082099,15.29,17.45,0.00
9,2022-10,3585624,3553183,14.99,16.93,0.00


In [7]:
# Filas que quitó (o marcó) cada regla, sumando los 24 meses
log = pd.read_csv(LOG_PATH)
log.groupby(["tipo", "regla"], sort=False)["filas"].sum().to_frame()

filas
tipo                regla                                                   
elimina fila        Fecha fuera del mes                                 2194
                    Filas completamente duplicadas                         3
                    duration_min <= 1 o > 180                        1077618
                    trip_distance < 0.1 o > 100                       773890
                    avg_speed_mph < 1 o > 80                           33438
solo tarifa (marca) RatecodeID = 5                                    189588
                    fare_amount < bajada de bandera ($2.50 / $3.00)   501017
                    fare_amount > 500                                    190

## 6. (Opcional) Borrar los archivos crudos

In [8]:
if BORRAR_CRUDOS_AL_FINAL:
    shutil.rmtree(RAW_DIR)
    print("Se borró data/raw/")
else:
    print("data/raw/ se conserva. Pon BORRAR_CRUDOS_AL_FINAL = True para liberar espacio.")

data/raw/ se conserva. Pon BORRAR_CRUDOS_AL_FINAL = True para liberar espacio.


## Cómo usar el archivo (para el resto del equipo)

El archivo completo tiene decenas de millones de filas. **Lee solo las columnas y filas que necesites:**

```python
import pandas as pd

RUTA = "data/processed/yellow_2022_2023_clean.parquet"
FEATURES = ["PULocationID", "DOLocationID", "pickup_hour", "pickup_dow",
            "is_airport_trip", "morning_rush", "night_rush", "weekend_rush"]

# Modelo de duración: todas las filas
duracion = pd.read_parquet(RUTA, columns=["tpep_pickup_datetime", *FEATURES, "trip_duration_min"])

# Modelo de tarifa: solo filas válidas para tarifa (+ new_tariff si el equipo lo usa)
tarifa = pd.read_parquet(
    RUTA,
    columns=["tpep_pickup_datetime", *FEATURES, "new_tariff", "fare_amount"],
    filters=[("valid_for_fare", "==", True)],
)

# Solo un periodo, por ejemplo 2023
anio_2023 = pd.read_parquet(RUTA, filters=[("tpep_pickup_datetime", ">=", pd.Timestamp("2023-01-01"))])
```

- La división train / validación / test **no está hecha**: el equipo la decide con `tpep_pickup_datetime`. Debe ser **por fechas**, nunca al azar.
- `trip_distance` no es input del modelo (data leakage).